# core

> Claude code api backend for fastllm

FastLLM's `claude_code` provider, as a thin adapter over `fastclaude`: `claude_mk_payload` hands the canonical history and tool schemas to `astream`, and `claude_acollect_stream` adapts the run's raw events to FastLLM's delta stream. FastLLM owns the tool loop, exactly as with every other provider: a reply that calls a tool ends the completion at the `tool_use`, the chat's loop executes it, and the next request's trailing `ToolResult` continues via fastclaude's deferral. Closing the stream mid-turn reaches `ClaudeRun.aclose`, which interrupts natively and reaps the process.


In [ ]:
#| default_exp core

In [ ]:
#| export
from fastcore.utils import *
from fastllm.types import *
from aidialog.msg_parts import ToolUse
from fastllm.anthropic import norm_sse_event, norm_tool_calls, norm_parts, norm_finish, norm_usage, finalize_usage, delta_index_fn, cost
from fastllm.streaming import mk_acollect_stream, Delta
from fastspec.errors import APIError
from fastclaude.core import astream, unqual, SERVER_TOOLS


In [ ]:
from fastllm.chat import AsyncChat, lite_mk_func, mk_msgs
from fastcore.test import *


`claude_mk_payload` passes the canonical history straight through: `astream` compiles, files, and resumes it itself, and a history ending in tool results becomes the deferred continuation with no work here. FastLLM's OpenAI-shaped tool schemas map to fastclaude's, and `web_search_options` enables Claude Code's own search tools:


In [ ]:
#| export
def claude_mk_payload(msgs, model, stream=False, **kwargs):
    "Build `astream` inputs: canonical history, tool schemas, native search tools"
    tools = [dict(name=f['name'], description=f.get('description',''), inputSchema=f.get('parameters', {}))
        for t in (kwargs.get('tools') or []) if (f := t.get('function'))]
    native = SERVER_TOOLS if kwargs.get('web_search_options') is not None else ()
    return dict(msgs=list(msgs), model=model, system=kwargs.get('system') or '', tools=tools or None, native_tools=native)


In [ ]:
def simple_add(a: int, b: int) -> int:
    "Add two numbers"
    return a + b

p = claude_mk_payload(mk_msgs(['What is 2+2?']), 'claude-sonnet-5', tools=[lite_mk_func(simple_add)])
test_eq(p['native_tools'], ())
test_eq(claude_mk_payload([], 'm', web_search_options='l')['native_tools'], SERVER_TOOLS)
p['tools'][0]


The stream adapter re-indexes partial events onto one global block sequence (`_reidx`), because thinking, text, and the tool call arrive as separate wire messages that each restart at index 0, then normalizes them to FastLLM `Delta`s with tool names unqualified. The calls are not marked server-executed: they are the chat's own tools, and FastLLM's loop must run them. One CLI quirk needs smoothing: a zero-argument call streams a single empty `partial_json` chunk, which never assembles into a `ToolUse` part, so a call block that closes without completing its JSON is synthesized with empty arguments:


In [ ]:
#| export
def _reidx():
    "Stateful rebase of per-message block indices onto one global sequence"
    base,mx = 0,-1
    def f(ev):
        nonlocal base,mx
        t = ev.get('type')
        if t=='message_start': base,mx = base+mx+1,-1
        elif t in ('content_block_start','content_block_delta','content_block_stop') and 'index' in ev:
            mx = max(mx, ev['index'])
            ev = {**ev, 'index': ev['index']+base}
        return ev
    return f


The whole adapter is one generator pipeline: raw run events filtered to partials, re-indexed, normalized, collected by FastLLM's standard collector. An error result raises as a provider `APIError` after the stream completes, and the run is always closed:


In [ ]:
#| export
async def claude_acollect_stream(payload, **kwargs):
    "Adapt one `ClaudeRun`'s raw events to FastLLM's delta stream; the chat's own loop executes tool calls"
    run,f = astream(**payload),_reidx()
    async def _deltas():
        opens,saw = {},set()
        async for m in run:
            if m.get('type')!='stream_event': continue
            ev = f(m['event'])
            et,idx = ev.get('type'), ev.get('index')
            d = norm_sse_event(ev)
            for tc in (d.tool_calls or []): tc.name = unqual(tc.name)
            if et=='content_block_start' and d.tool_calls: opens[idx] = d.tool_calls[0]
            elif et=='content_block_delta' and d.tool_calls and (nested_idx(ev, 'delta', 'partial_json') or '').endswith('}'): saw.add(idx)
            yield d
            if et=='content_block_stop' and (tc := opens.pop(idx, None)) is not None and idx not in saw:
                yield Delta(tool_calls=[ToolUse(id=tc.id, name=tc.name, arguments={})], raw=dict(index=idx))
    try:
        async for o in mk_acollect_stream(_deltas(), index_fn=delta_index_fn, api_name='claude_code', **kwargs): yield o
        if run.result and run.result.get('is_error'):
            raise APIError(str(run.result.get('result') or run.result.get('subtype')), provider='claude_code',
                model=payload.get('model'), status_code=run.result.get('api_error_status'), raw=run.result)
    finally: await run.aclose()


In [ ]:
#| export
api_registry.register('claude_code',
    norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage,
    finalize_usage=finalize_usage, mk_payload=claude_mk_payload, acollect_stream=claude_acollect_stream, cost=cost)

## Live runs

Against the real CLI (genuine captured outputs; spends tokens, so out of automated runs): a chat with one Python tool runs the whole loop from FastLLM's side. The first request stops at the call, FastLLM executes `simple_add` itself, and the continuation request carries the result back through the deferral. One `chat()` call, two claude processes, and the familiar history shape with the real result in its `{.tool}` block:


In [ ]:
#| eval: false
chat = AsyncChat('claude_code/claude-sonnet-5', tools=[simple_add])
rs = await chat('What is 7+3? Use the tool, then answer with only the number.', stream=True, max_steps=3)
parts = [o async for o in rs]
test_eq([m.role for m in chat.hist], ['user','assistant','tool','assistant'])
test_eq(chat.hist[-1].text, '10')
test('"result": "10"', chat.full(), in_)
[type(o).__name__ for o in parts]


['ToolUse', 'Completion', 'ToolResult', 'Refresh', 'Text', 'Completion']